In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cpu
CUDA available: False


In [ ]:
# Training text: a small toy dataset
text = """
i love machine learning
i love deep learning
machine learning is interesting
deep learning uses neural networks
neural networks learn from data
machine learning learns patterns
i study artificial intelligence
artificial intelligence uses machine learning
deep learning is a part of machine learning
"""

print(text)
print("Number of characters:", len(text))


i love machine learning
i love deep learning
machine learning is interesting
deep learning uses neural networks
neural networks learn from data
machine learning learns patterns
i study artificial intelligence
artificial intelligence uses machine learning
deep learning is a part of machine learning

Number of characters: 300


In [ ]:
# Split the full text into tokens
tokens=text.split()
print('num of tokens: ',len(tokens))

# Build the vocabulary
vocab =sorted(set(tokens))
print(vocab )
print(f"length of vocab : {len(vocab)}")

num of tokens:  43
['a', 'artificial', 'data', 'deep', 'from', 'i', 'intelligence', 'interesting', 'is', 'learn', 'learning', 'learns', 'love', 'machine', 'networks', 'neural', 'of', 'part', 'patterns', 'study', 'uses']
length of vocab : 21


In [ ]:
# Create mappings between tokens and IDs

word_to_id={word:id for id,word in enumerate(vocab)}
id_to_word={id:word for word, id in word_to_id.items()}

# Encode the full text
token_ids=[word_to_id[word] for word in tokens]

print("First 20 tokens:", tokens[:20])
print("Vocabulary size:", len(vocab))
print("First 20 token IDs:", token_ids[:20])

First 20 tokens: ['i', 'love', 'machine', 'learning', 'i', 'love', 'deep', 'learning', 'machine', 'learning', 'is', 'interesting', 'deep', 'learning', 'uses', 'neural', 'networks', 'neural', 'networks', 'learn']
Vocabulary size: 21
First 20 token IDs: [5, 12, 13, 10, 5, 12, 3, 10, 13, 10, 8, 7, 3, 10, 20, 15, 14, 15, 14, 9]


In [ ]:
# Convert token IDs to a PyTorch tensor
data = torch.tensor(token_ids, dtype=torch.long)

# Define the context length
context_length = 4

# Create input and target sequences
x = data[:-1]
y = data[1:]

# Build training examples
X = []
Y = []

for i in range(len(x) - context_length + 1):
    X.append(x[i:i + context_length])
    Y.append(y[i:i + context_length])

X = torch.stack(X)
Y = torch.stack(Y)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: torch.Size([39, 4])
Y shape: torch.Size([39, 4])


In [ ]:
for x,y in enumerate(zip(X,Y)):
    print(x,y)

0 (tensor([ 5, 12, 13, 10]), tensor([12, 13, 10,  5]))
1 (tensor([12, 13, 10,  5]), tensor([13, 10,  5, 12]))
2 (tensor([13, 10,  5, 12]), tensor([10,  5, 12,  3]))
3 (tensor([10,  5, 12,  3]), tensor([ 5, 12,  3, 10]))
4 (tensor([ 5, 12,  3, 10]), tensor([12,  3, 10, 13]))
5 (tensor([12,  3, 10, 13]), tensor([ 3, 10, 13, 10]))
6 (tensor([ 3, 10, 13, 10]), tensor([10, 13, 10,  8]))
7 (tensor([10, 13, 10,  8]), tensor([13, 10,  8,  7]))
8 (tensor([13, 10,  8,  7]), tensor([10,  8,  7,  3]))
9 (tensor([10,  8,  7,  3]), tensor([ 8,  7,  3, 10]))
10 (tensor([ 8,  7,  3, 10]), tensor([ 7,  3, 10, 20]))
11 (tensor([ 7,  3, 10, 20]), tensor([ 3, 10, 20, 15]))
12 (tensor([ 3, 10, 20, 15]), tensor([10, 20, 15, 14]))
13 (tensor([10, 20, 15, 14]), tensor([20, 15, 14, 15]))
14 (tensor([20, 15, 14, 15]), tensor([15, 14, 15, 14]))
15 (tensor([15, 14, 15, 14]), tensor([14, 15, 14,  9]))
16 (tensor([14, 15, 14,  9]), tensor([15, 14,  9,  4]))
17 (tensor([15, 14,  9,  4]), tensor([14,  9,  4,  2]))
18

In [ ]:
# Display the first training example
print("Input IDs: ", X[0].tolist())
print("Target IDs:", Y[0].tolist())

# Convert IDs back to words
input_words = [id_to_word[token_id] for token_id in X[0].tolist()]
target_words = [id_to_word[token_id] for token_id in Y[0].tolist()]

print("Input words: ", input_words)
print("Target words:", target_words)

Input IDs:  [5, 12, 13, 10]
Target IDs: [12, 13, 10, 5]
Input words:  ['i', 'love', 'machine', 'learning']
Target words: ['love', 'machine', 'learning', 'i']


In [ ]:
# Define model dimensions
vocab_size = len(vocab)
embed_dim = 16

# Create the embedding layer
embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embed_dim
)

# Convert the first input sequence into embeddings
sample_ids = X[0]
sample_embeddings = embedding(sample_ids)

print("Input shape:", sample_ids.shape)
print("Embedding shape:", sample_embeddings.shape)
print("\nFirst token embedding:")
print(sample_embeddings[0])

Input shape: torch.Size([4])
Embedding shape: torch.Size([4, 16])

First token embedding:
tensor([ 0.4262, -0.0825,  0.9073,  0.6967,  0.0053, -0.2403, -0.7090, -0.4782,
        -0.5032,  0.1082,  0.9141,  0.2008,  1.2093,  1.1652, -0.7820, -0.2452],
       grad_fn=<SelectBackward0>)


In [ ]:
import math

def positional_encoding(seq_len, embed_dim):
    pe = torch.zeros(seq_len, embed_dim)

    for pos in range(seq_len):
        for i in range(0, embed_dim, 2):
            angle = pos / (10000 ** (i / embed_dim))

            pe[pos, i] = math.sin(angle)
            pe[pos, i + 1] = math.cos(angle)

    return pe

# Create positional encodings for our sequence
seq_len = X[0].shape[0]

pe = positional_encoding(seq_len, embed_dim)

print("Positional Encoding shape:", pe.shape)
print(pe)

Positional Encoding shape: torch.Size([4, 16])
tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  3.1098e-01,  9.5042e-01,  9.9833e-02,
          9.9500e-01,  3.1618e-02,  9.9950e-01,  9.9998e-03,  9.9995e-01,
          3.1623e-03,  9.9999e-01,  1.0000e-03,  1.0000e+00,  3.1623e-04,
          1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  5.9113e-01,  8.0658e-01,  1.9867e-01,
          9.8007e-01,  6.3203e-02,  9.9800e-01,  1.9999e-02,  9.9980e-01,
          6.3245e-03,  9.9998e-01,  2.0000e-03,  1.0000e+00,  6.3246e-04,
          1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  8.1265e-01,  5.8275e-01,  2.9552e-01,
          9.5534e-01,  9.4726e-02,  9.9550e-01,  2.9996e-02,  9.9955e-01,
          9.4867e-03,  9.9995e-01,  3.0000e-03,  1.0000e+00,  9.4868e-

In [ ]:
# Add positional information to token embeddings
position_aware_embeddings = sample_embeddings + pe

print("Token embeddings shape:", sample_embeddings.shape)
print("Positional encoding shape:", pe.shape)
print("Final shape:", position_aware_embeddings.shape)

Token embeddings shape: torch.Size([4, 16])
Positional encoding shape: torch.Size([4, 16])
Final shape: torch.Size([4, 16])


In [ ]:
# Input to self-attention
z = position_aware_embeddings

# Create learnable linear projections
query_layer = nn.Linear(embed_dim, embed_dim)
key_layer = nn.Linear(embed_dim, embed_dim)
value_layer = nn.Linear(embed_dim, embed_dim)

# Generate Q, K, V
Q = query_layer(z)
K = key_layer(z)
V = value_layer(z)

print("Input shape:", z.shape)
print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

Input shape: torch.Size([4, 16])
Q shape: torch.Size([4, 16])
K shape: torch.Size([4, 16])
V shape: torch.Size([4, 16])


In [ ]:
# Calculate attention scores
scores = Q @ K.T

print("Scores shape:", scores.shape)
print(scores)

Scores shape: torch.Size([4, 4])
tensor([[-0.3929, -1.1894, -1.7479,  0.1490],
        [-0.5820, -0.5753, -0.4078, -1.0549],
        [ 0.8888, -0.3075,  0.2296, -0.4139],
        [ 0.7795,  0.6574,  0.8431,  0.2275]], grad_fn=<MmBackward0>)


In [ ]:
import math

# Scale the attention scores
scaled_scores = scores / math.sqrt(embed_dim)

# Create a causal mask
seq_len = scaled_scores.shape[0]

mask = torch.tril(
    torch.ones(seq_len, seq_len, dtype=torch.bool)
)

# Block future positions
masked_scores = scaled_scores.masked_fill(
    ~mask,
    float("-inf")
)

print("Scaled scores:")
print(scaled_scores)

print("\nMasked scores:")
print(masked_scores)

Scaled scores:
tensor([[-0.0982, -0.2974, -0.4370,  0.0373],
        [-0.1455, -0.1438, -0.1019, -0.2637],
        [ 0.2222, -0.0769,  0.0574, -0.1035],
        [ 0.1949,  0.1644,  0.2108,  0.0569]], grad_fn=<DivBackward0>)

Masked scores:
tensor([[-0.0982,    -inf,    -inf,    -inf],
        [-0.1455, -0.1438,    -inf,    -inf],
        [ 0.2222, -0.0769,  0.0574,    -inf],
        [ 0.1949,  0.1644,  0.2108,  0.0569]], grad_fn=<MaskedFillBackward0>)


In [ ]:
# Convert masked scores into attention weights
attention_weights = F.softmax(
    masked_scores,
    dim=-1
)

print("Attention weights:")
print(attention_weights)

print("\nRow sums:")
print(attention_weights.sum(dim=-1))

Attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4996, 0.5004, 0.0000, 0.0000],
        [0.3862, 0.2863, 0.3275, 0.0000],
        [0.2593, 0.2515, 0.2634, 0.2258]], grad_fn=<SoftmaxBackward0>)

Row sums:
tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [ ]:
# Calculate the attention output
attention_output = attention_weights @ V

print("Attention output shape:", attention_output.shape)
print("\nAttention output:")
print(attention_output)

Attention output shape: torch.Size([4, 16])

Attention output:
tensor([[-0.5903,  0.2309,  0.1978,  1.4437,  0.9518, -0.5598, -0.3718, -1.5263,
          1.0086,  0.1846, -0.1911, -0.1905, -0.9724,  0.5848, -0.0209,  0.7546],
        [ 0.0566,  0.6962,  0.2014,  0.4366,  0.7951, -0.2962, -0.1352, -0.8800,
         -0.0406,  0.6603, -0.1559, -0.5094, -1.2169,  0.4281,  0.2864,  0.9643],
        [-0.1655,  0.4425,  0.2324,  0.5292,  0.7394, -0.3342, -0.3267, -0.8818,
          0.3996,  0.5440, -0.1238, -0.3252, -0.7594,  0.5268,  0.2966,  0.5123],
        [-0.3251,  0.3194,  0.1259,  0.4665,  0.4962, -0.4553, -0.1385, -0.8611,
          0.5875,  0.2620, -0.0604, -0.3662, -0.5106,  0.3892,  0.2197,  0.0971]],
       grad_fn=<MmBackward0>)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()

        # 1. Create Query, Key, and Value projections
        self.query_layer = nn.Linear(embed_dim, embed_dim)
        self.key_layer = nn.Linear(embed_dim, embed_dim)
        self.value_layer = nn.Linear(embed_dim, embed_dim)

        # 2. Output projection
        self.output_layer = nn.Linear(embed_dim, embed_dim)

        # 3. Scaling factor
        self.scale = math.sqrt(embed_dim)

    def forward(self, x):
        # x shape: (batch_size, seq_len, embed_dim)

        batch_size, seq_len, embed_dim = x.shape

        # Step 1: Calculate Q, K, V
        Q = self.query_layer(x)
        K = self.key_layer(x)
        V = self.value_layer(x)

        # Step 2: Calculate attention scores
        scores = Q @ K.transpose(-2, -1)

        # Step 3: Scale the scores
        scores = scores / self.scale

        # Step 4: Create causal mask
        mask = torch.tril(
            torch.ones(seq_len, seq_len, device=x.device)
        ).bool()

        # Step 5: Prevent looking at future tokens
        scores = scores.masked_fill(
            ~mask,
            float("-inf")
        )

        # Step 6: Convert scores to attention weights
        attention_weights = F.softmax(scores, dim=-1)

        # Step 7: Combine Value vectors
        attention_output = attention_weights @ V

        # Step 8: Apply output projection
        output = self.output_layer(attention_output)

        return output, attention_weights


# Create the layer
attention_layer = CausalSelfAttention(embed_dim=16)

# Prepare input: (batch_size, seq_len, embed_dim)
x = position_aware_embeddings.unsqueeze(0)

# Forward pass
output, attention_weights = attention_layer(x)

# Display results
print("Input shape:", x.shape)
print("Attention weights shape:", attention_weights.shape)
print("Output shape:", output.shape)

print("\nAttention weights:")
print(attention_weights[0])

print("\nAttention output:")
print(output[0])

Input shape: torch.Size([1, 4, 16])
Attention weights shape: torch.Size([1, 4, 4])
Output shape: torch.Size([1, 4, 16])

Attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5489, 0.4511, 0.0000, 0.0000],
        [0.3400, 0.3022, 0.3579, 0.0000],
        [0.2893, 0.2310, 0.2344, 0.2454]], grad_fn=<SelectBackward0>)

Attention output:
tensor([[-0.3851,  0.0929, -0.0544,  0.3514, -0.4478,  0.1246, -0.4283,  0.4412,
          0.0360, -0.2918,  0.3252, -0.4096,  0.0431,  0.7014, -0.3441, -0.0127],
        [-0.3370,  0.2195,  0.2043,  0.1237, -0.6264,  0.4026, -0.4396,  0.5677,
          0.0725, -0.1613,  0.3175, -0.6881, -0.1163,  0.7243, -0.4421, -0.2197],
        [-0.2232,  0.0377,  0.2643, -0.0182, -0.5612,  0.3367, -0.3824,  0.4470,
          0.1306, -0.1966,  0.2421, -0.5344,  0.0251,  0.5001, -0.3134, -0.1643],
        [-0.2637,  0.0401,  0.2541,  0.0228, -0.4751,  0.1974, -0.3837,  0.3275,
          0.1416, -0.1677,  0.2331, -0.3710,  0.0431,  0.3973, -0.2637, -0.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()

        # 1. Causal Self-Attention
        self.attention = CausalSelfAttention(embed_dim)

        # 2. Layer Normalization
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        # 3. Feed-Forward Network
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

    def forward(self, x):
        # Self-Attention
        attention_output, attention_weights = self.attention(x)

        # Residual Connection + LayerNorm
        x = self.norm1(x + attention_output)

        # Feed-Forward Network
        ff_output = self.feed_forward(x)

        # Residual Connection + LayerNorm
        x = self.norm2(x + ff_output)

        return x, attention_weights


# Create the Transformer Block
transformer_block = TransformerBlock(embed_dim=16)

# Forward pass using our previous input
block_output, block_attention_weights = transformer_block(x)

# Check shapes
print("Input shape:", x.shape)
print("Block output shape:", block_output.shape)
print("Attention weights shape:", block_attention_weights.shape)

Input shape: torch.Size([1, 4, 16])
Block output shape: torch.Size([1, 4, 16])
Attention weights shape: torch.Size([1, 4, 4])


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        context_length
    ):
        super().__init__()

        # Token embeddings
        self.token_embedding = nn.Embedding(
            vocab_size, embed_dim
        )

        # Learnable positional embeddings
        self.position_embedding = nn.Embedding(
            context_length, embed_dim
        )

        # Transformer Block
        self.transformer_block = TransformerBlock(embed_dim)

        # Predict vocabulary scores
        self.output_layer = nn.Linear(
            embed_dim, vocab_size
        )

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape

        # Token embeddings
        token_embeddings = self.token_embedding(input_ids)

        # Position embeddings
        positions = torch.arange(
            seq_len, device=input_ids.device
        )
        position_embeddings = self.position_embedding(positions)

        # Combine token and position information
        x = token_embeddings + position_embeddings

        # Transformer Block
        x, attention_weights = self.transformer_block(x)

        # Predict scores for every vocabulary word
        logits = self.output_layer(x)

        return logits, attention_weights


# Create the model
model = MiniGPT(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    context_length=context_length
)

# Test one input sequence
sample_input = X[0].unsqueeze(0)

logits, attention_weights = model(sample_input)

print("Input shape:", sample_input.shape)
print("Logits shape:", logits.shape)
print("Attention weights shape:", attention_weights.shape)

Input shape: torch.Size([1, 4])
Logits shape: torch.Size([1, 4, 21])
Attention weights shape: torch.Size([1, 4, 4])


In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.003
)

# Training settings
epochs = 300

for epoch in range(epochs):

    # Training mode
    model.train()

    # Reset gradients
    optimizer.zero_grad()

    # Forward pass
    logits, _ = model(X)

    # Reshape predictions and targets
    loss = criterion(
        logits.reshape(-1, vocab_size),
        Y.reshape(-1)
    )

    # Backpropagation
    loss.backward()

    # Update model parameters
    optimizer.step()

    # Print progress
    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch [{epoch + 1}/{epochs}], "
            f"Loss: {loss.item():.4f}"
        )

Epoch [50/300], Loss: 1.0731
Epoch [100/300], Loss: 0.5386
Epoch [150/300], Loss: 0.4246
Epoch [200/300], Loss: 0.3905
Epoch [250/300], Loss: 0.3653
Epoch [300/300], Loss: 0.3411


In [ ]:
# Switch to evaluation mode
model.eval()

with torch.no_grad():
    sample_input = X[0].unsqueeze(0)

    logits, _ = model(sample_input)

    # Select the most likely word at each position
    predicted_ids = logits.argmax(dim=-1)[0]

    # Get the correct target IDs
    actual_ids = Y[0]

    # Convert IDs to words
    input_words = [id_to_word[i.item()] for i in X[0]]
    predicted_words = [
        id_to_word[i.item()] for i in predicted_ids
    ]
    actual_words = [
        id_to_word[i.item()] for i in actual_ids
    ]

    print("Input words:    ", input_words)
    print("Predicted words:", predicted_words)
    print("Actual words:   ", actual_words)

    accuracy = (
        (predicted_ids == actual_ids).float().mean().item()
    )

    print(f"\nSequence Accuracy: {accuracy:.2%}")

Input words:     ['i', 'love', 'machine', 'learning']
Predicted words: ['love', 'machine', 'learning', 'i']
Actual words:    ['love', 'machine', 'learning', 'i']

Sequence Accuracy: 100.00%


In [ ]:
def generate_text(model, prompt, max_new_tokens=10):
    model.eval()

    # Convert prompt words to token IDs
    words = prompt.split()
    token_ids = [
        word_to_id[word] for word in words
        if word in word_to_id
    ]

    if not token_ids:
        raise ValueError("The prompt has no known vocabulary words.")

    generated = token_ids.copy()

    with torch.no_grad():
        for _ in range(max_new_tokens):

            # Keep only the latest context_length tokens
            context = generated[-context_length:]

            input_ids = torch.tensor(
                [context],
                dtype=torch.long
            )

            # Predict next-token scores
            logits, _ = model(input_ids)

            # Get scores at the last position
            next_token_logits = logits[0, -1, :]

            # Select the most likely next token
            next_token_id = next_token_logits.argmax().item()

            # Append the predicted token
            generated.append(next_token_id)

    return " ".join(
        id_to_word[token_id] for token_id in generated
    )


print(generate_text(model, "i love", max_new_tokens=10))

i love machine learning i love deep learning machine learning is interesting


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        context_length,
        num_blocks=2
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size, embed_dim
        )

        self.position_embedding = nn.Embedding(
            context_length, embed_dim
        )

        # Stack multiple Transformer Blocks
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(embed_dim)
            for _ in range(num_blocks)
        ])

        self.output_layer = nn.Linear(
            embed_dim, vocab_size
        )

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape

        # Token embeddings
        x = self.token_embedding(input_ids)

        # Positional embeddings
        positions = torch.arange(
            seq_len, device=input_ids.device
        )

        x = x + self.position_embedding(positions)

        # Pass through all Transformer Blocks
        attention_weights = []

        for block in self.transformer_blocks:
            x, weights = block(x)
            attention_weights.append(weights)

        # Predict next-token scores
        logits = self.output_layer(x)

        return logits, attention_weights


# Create a NEW model with two Transformer Blocks
model = MiniGPT(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    context_length=context_length,
    num_blocks=2
)

# Test the model
sample_input = X[0].unsqueeze(0)

logits, attention_weights = model(sample_input)

print("Input shape:", sample_input.shape)
print("Logits shape:", logits.shape)
print("Number of Transformer Blocks:",
      len(model.transformer_blocks))
print("Number of Attention maps:",
      len(attention_weights))

Input shape: torch.Size([1, 4])
Logits shape: torch.Size([1, 4, 21])
Number of Transformer Blocks: 2
Number of Attention maps: 2


In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer for the NEW model
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.003
)

epochs = 300

for epoch in range(epochs):
    model.train()

    # Clear previous gradients
    optimizer.zero_grad()

    # Forward pass
    logits, _ = model(X)

    # Calculate next-token prediction loss
    loss = criterion(
        logits.reshape(-1, vocab_size),
        Y.reshape(-1)
    )

    # Backpropagation
    loss.backward()

    # Update parameters
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch [{epoch + 1}/{epochs}], "
            f"Loss: {loss.item():.4f}"
        )

Epoch [50/300], Loss: 1.0125
Epoch [100/300], Loss: 0.3624
Epoch [150/300], Loss: 0.2717
Epoch [200/300], Loss: 0.2519
Epoch [250/300], Loss: 0.2415
Epoch [300/300], Loss: 0.2370


In [ ]:
model.eval()

with torch.no_grad():
    logits, _ = model(X)

    # Predicted next-token IDs
    predictions = logits.argmax(dim=-1)

    # Compare predictions with targets
    correct = (predictions == Y).sum().item()
    total = Y.numel()

    accuracy = correct / total

    print("Correct predictions:", correct)
    print("Total predictions:", total)
    print(f"Training Accuracy: {accuracy:.2%}")

Correct predictions: 136
Total predictions: 156
Training Accuracy: 87.18%
